A file for computing the "correct" scale factor for Sfm calibrations, given precise extrinsics.

In [41]:
import numpy as np
import os
import json
u = lambda x: f"cam_{(x + 1):02d}"

In [44]:
def read_extrinsics(extrinsics_path):
    config = {}
    camkeys = set([f"cam_{i:02}" for i in range(1, 13)]) # ignore 13 camera
    with open(extrinsics_path) as f:
        extrinsics_str = f.readlines()
    for line in extrinsics_str:
        if line.lstrip().startswith("#"):
            continue
        else:
            w = line.split()
            if len(w) == 0 or w[0] not in camkeys:
                continue
            rotation = np.array(list(map(lambda x: float(x), w[7:]))).reshape((3, 3), order="C")
            position = np.array(list(map(lambda x: float(x), w[1:4])))
            config[w[0]] = {
                "extrinsics": {
                    "rotation": (rotation).tolist(),
                    "center": (position).tolist()
                }
            }
    if len(camkeys.intersection(config.keys())) != len(camkeys):
        raise Exception("extrinsics file does not contain extrinsics for all cameras")
    return config

In [11]:
path = "../assets/fip_calibration/extrinsics"
confs = []
for file in os.listdir(path):
    if os.path.splitext(file)[1] == ".txt":
        confs.append(read_extrinsics(os.path.join(path, file)))

In [65]:
rel_dist = np.zeros((len(confs[0]), len(confs[0]), len(confs)))

for pos_set in range(len(confs)):
    for i in range(len(confs[0])):
        for j in range(len(confs[0])):
            p1 = np.array(confs[pos_set][u(i)]["extrinsics"]["center"])
            p2 = np.array(confs[pos_set][u(j)]["extrinsics"]["center"])
            dist = np.linalg.norm(p1 - p2)
            rel_dist[i, j, pos_set] = dist

min_values = np.min(rel_dist, axis=2)
max_values = np.max(rel_dist, axis=2)
max_def = np.abs(max_values - min_values)
mean_dist = rel_dist.mean(axis=2)
mean_dist[mean_dist == 0] = 0.0001
max_error = max_def / mean_dist

In [68]:
def create_scaled_conf(infile):
    o = os.path.split(infile)
    outfile = os.path.join(o[0], os.path.splitext(o[1])[0] + "_scaled.json")

    with open(infile) as f:
        d = json.load(f)
    points = []
    for i in range(0, 12):
        points.append(np.array(d[u(i) + ".png"]["extrinsics"]["center"]))
    scales = []
    for i in range(0, 12):
        for j in range(0, 12):
            if i == j:
                continue
            s = rel_dist[i, j] / np.linalg.norm(points[i] - points[j])
            scales.append(s)
    scale = np.array(scales).mean()
    diff = np.array(scales).max() - np.array(scales).min()
    print(f"err {diff}")
    print(f"Scale {scale}")

    points = np.stack(points, axis=0)
    points = points * scale

    for i in range(0, 12):
        d[u(i) + ".png"]["extrinsics"]["center"] = points[i].tolist()

    with open(outfile, "w") as f:
        json.dump(d, f, indent=4)

create_scaled_conf(infile="../assets/poses/2024_06_10_14_26_Lot3.json")

err 0.11168402361188257
Scale 1.2824026922984582
